# 문항 1 순차 · 스레드 · 프로세스 속도 비교

In [ ]:
# %pip install httpx
# %pip install Flask

In [ ]:
# bench_server.py

from flask import Flask, jsonify
import time, random

app = Flask(__name__)

@app.get("/item/<int:n>")
def item(n):
    time.sleep(random.uniform(0.3, 0.6))     # 네트워크 대기를 흉내낸다
    return jsonify({"id": n, "name": f"item-{n}"})

if __name__ == "__main__":
    app.run(port=5000, threaded=True)

In [ ]:
# first_assignment.py

from multiprocessing.dummy import Pool as dummyPool
from multiprocessing import Pool
from urllib import parse
import requests
import time

def fetch(url):
    r = requests.get(url, timeout=10)
    return r.json()

# 순차
def sequential(urls):
    start = time.time()
    results = []
    for url in urls:
        results.append(fetch(url))
    end = time.time()
    time_taken = end - start
    return results, time_taken


# 스레드
def threaded(urls, worker):
    start = time.time()
    with dummyPool(worker) as pool:
        results = pool.map(fetch, urls)
    end = time.time()
    time_taken = end - start
    return results, time_taken

# 프로세스
def processed(urls, worker):
    start = time.time()
    with Pool(worker) as pool:
        results = pool.map(fetch, urls)
    end = time.time()
    time_taken = end - start
    return results, time_taken

if __name__ == "__main__":
    workers = [1, 3, 5, 10]
    urls = [f"http://127.0.0.1:5000/item/{i}" for i in range(1, 51)]

    # Sequential
    print("순차적으로 요청을 보내는 경우")
    results, time_taken = sequential(urls)
    print(f"Sequential: {len(results)} items fetched in {time_taken:.2f} seconds.")

    # Thread
    print("스레드를 이용하여 요청을 보내는 경우")
    for worker in workers:
        results, time_taken = threaded(urls, worker)
        print(f"Threaded ({worker} workers): {len(results)} items fetched in {time_taken:.2f} seconds.")

    # Process
    print("프로세스를 이용하여 요청을 보내는 경우")
    for worker in workers:
        results, time_taken = processed(urls, worker)
        print(f"Processed ({worker} workers): {len(results)} items fetched in {time_taken:.2f} seconds.")



### Output 

순차적으로 요청을 보내는 경우

Sequential: 50 items fetched in 23.92 seconds.

스레드를 이용하여 요청을 보내는 경우

Threaded (1 workers): 50 items fetched in 24.15 seconds.

Threaded (3 workers): 50 items fetched in 8.74 seconds.

Threaded (5 workers): 50 items fetched in 5.23 seconds.

Threaded (10 workers): 50 items fetched in 2.75 seconds.

프로세스를 이용하여 요청을 보내는 경우

Processed (1 workers): 50 items fetched in 25.03 seconds.

Processed (3 workers): 50 items fetched in 9.63 seconds.

Processed (5 workers): 50 items fetched in 6.71 seconds.

Processed (10 workers): 50 items fetched in 4.79 seconds.

## Sequential
* 23.92초

## 스레드 & 프로세스

| 워커 수 | 스레드 | 프로세스 |
|:------ | :---- | :------ |
| 1      | 24.15 | 25.03   |
| 3      | 8.74  | 9.63    |
| 5      | 5.23  | 6.71    |
| 10     | 2.75  | 4.79    |

1. 차이가 나는것은 메모리의 사용방식 입니다.
    - 스레드는 메모리를 공유하지만 프로세스는 따로 사용합니다.

2. 풀 생성 비용 또한 차이가 납니다.
    - 스레드는 생성 비용이 작지만 프로세스는 큽니다.

3.  대기 시간은 거의 비슷합니다.

# 문항 2 Scrapy 포팅 또는 httpx 비동기 전환 (택일)

### A Process

1. scrapy startproject quotes

2. cd quotes

3. scrapy genspider myquotes quotes.toscrape.com

4. implemented parse() in myquotes.py

5. scrapy crawl myquotes -O out.csv

In [ ]:
# myquotes.py

import scrapy


class MyquotesSpider(scrapy.Spider):
    name = "myquotes"
    allowed_domains = ["quotes.toscrape.com"]
    start_urls = ["https://quotes.toscrape.com"]

    def parse(self, response):
        for quote in response.css("div.quote"):
            yield {
                "text": quote.css("span.text::text").get(),
                "author": quote.css("small.author::text").get(),
                "tags": quote.css("div.tags a.tag::text").getall(),
            }
        next_page = response.css("li.next a::attr(href)").get()
        if next_page:
            yield response.follow(next_page, self.parse)